In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("Verified_Train_Dataset.csv")

In [4]:
print(df.head())

   SN  Train_No Station_Code   1A   2A   3A   SL  Station_Name  Route_Number  \
0   1       107          SWV  100  100  100  100  SAWANTWADI R             1   
1   2       107         THVM  260  228  196  164        THIVIM             1   
2   3       107         KRMI  345  296  247  198       KARMALI             1   
3   4       107          MAO  490  412  334  256   MADGOAN JN.             1   
4   1       108          MAO  100  100  100  100   MADGOAN JN.             1   

  Arrival_time Departure_Time  Distance  
0     00:00:00       10:25:00         0  
1     11:06:00       11:08:00        32  
2     11:28:00       11:30:00        49  
3     12:10:00       00:00:00        78  
4     00:00:00       20:30:00         0  


In [5]:
df.head()

,SN,Train_No,Station_Code,1A,2A,3A,SL,Station_Name,Route_Number,Arrival_time,Departure_Time,Distance
0,1,107,SWV,100,100,100,100,SAWANTWADI R,1,00:00:00,10:25:00,0
1,2,107,THVM,260,228,196,164,THIVIM,1,11:06:00,11:08:00,32
2,3,107,KRMI,345,296,247,198,KARMALI,1,11:28:00,11:30:00,49
3,4,107,MAO,490,412,334,256,MADGOAN JN.,1,12:10:00,00:00:00,78
4,1,108,MAO,100,100,100,100,MADGOAN JN.,1,00:00:00,20:30:00,0


In [6]:
df.tail()

,SN,Train_No,Station_Code,1A,2A,3A,SL,Station_Name,Route_Number,Arrival_time,Departure_Time,Distance
186069,8,99908,AKRD,195,176,157,138,AKURDI,1,23:30:00,23:31:00,19
186070,9,99908,DEHR,220,196,172,148,DEHU ROAD,1,23:35:00,23:36:00,24
186071,10,99908,BGWI,240,212,184,156,BEGDAEWAI,1,23:39:00,23:40:00,28
186072,11,99908,GRWD,255,224,193,162,GHORAWADI,1,23:41:00,23:42:00,31
186073,12,99908,TGN,270,236,202,168,TALEGAON,1,23:50:00,00:00:00,34


In [7]:
df['Arrival_time'] = pd.to_datetime(
    df['Arrival_time'],
    errors='coerce'
)

df['Departure_Time'] = pd.to_datetime(
    df['Departure_Time'],
    errors='coerce'
)

/var/folders/0c/7f8p6mvs5mg8nbh9mllqtkb80000gn/T/ipykernel_9623/761387861.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Arrival_time'] = pd.to_datetime(
/var/folders/0c/7f8p6mvs5mg8nbh9mllqtkb80000gn/T/ipykernel_9623/761387861.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Departure_Time'] = pd.to_datetime(


In [8]:
def find_direct_trains(source, destination):

    trains = []

    for train_no in df['Train_No'].unique():

        route = (
            df[df['Train_No'] == train_no]
            .sort_values('SN')
        )

        stations = route['Station_Name'].str.upper().tolist()

        if (source.upper() in stations and
            destination.upper() in stations):

            src_index = stations.index(source.upper())
            dest_index = stations.index(destination.upper())

            # Destination must come after source
            if src_index < dest_index:

                src_row = route.iloc[src_index]
                dest_row = route.iloc[dest_index]

                duration = (
                    dest_row['Arrival_time']
                    - src_row['Departure_Time']
                ).total_seconds()/3600

                if duration < 0:
                    duration += 24

                trains.append({
                    'Train_No': train_no,
                    'Source': source,
                    'Destination': destination,
                    'Estimated_Duration(Hours)':
                        round(duration,2)
                })

    return pd.DataFrame(trains)

In [9]:
source_station = input(
    "Enter Source Station: "
)

destination_station = input(
    "Enter Destination Station: "
)

result = find_direct_trains(
    source_station,
    destination_station
)

if result.empty:
    print("\nNo Direct Trains Found.")
else:
    print("\nAvailable Direct Trains\n")
    print(result)

Enter Source Station:  MADGOAN JN.
Enter Destination Station:  SAWANTWADI R



Available Direct Trains

    Train_No       Source   Destination  Estimated_Duration(Hours)
0        108  MADGOAN JN.  SAWANTWADI R                       1.92
1        128  MADGOAN JN.  SAWANTWADI R                       1.60
2      10104  MADGOAN JN.  SAWANTWADI R                       1.43
3      10112  MADGOAN JN.  SAWANTWADI R                       1.60
4      11086  MADGOAN JN.  SAWANTWADI R                       1.37
5      12202  MADGOAN JN.  SAWANTWADI R                       1.50
6      12431  MADGOAN JN.  SAWANTWADI R                       1.00
7      12741  MADGOAN JN.  SAWANTWADI R                       1.20
8      16336  MADGOAN JN.  SAWANTWADI R                       1.33
9      22149  MADGOAN JN.  SAWANTWADI R                       1.10
10     50102  MADGOAN JN.  SAWANTWADI R                       1.65
11     50108  MADGOAN JN.  SAWANTWADI R                       2.17


In [10]:
def train_enquiry(source, destination):

    found = False

    for train_no in df['Train_No'].unique():

        route = (
            df[df['Train_No'] == train_no]
            .sort_values('SN')
        )

        stations = route['Station_Name'].str.upper().tolist()

        if (source.upper() in stations and
            destination.upper() in stations):

            s = stations.index(source.upper())
            d = stations.index(destination.upper())

            if s < d:

                found = True

                start = route.iloc[0]['Station_Name']
                end = route.iloc[-1]['Station_Name']

                duration = (
                    route.iloc[d]['Arrival_time']
                    - route.iloc[s]['Departure_Time']
                ).total_seconds()/3600

                if duration < 0:
                    duration += 24

                print("---------------------------------")
                print("Train Number :", train_no)
                print("Route        :", start,
                      "->", end)
                print("Journey      :", source,
                      "->", destination)
                print("Duration     :",
                      round(duration,2),
                      "Hours")

    if not found:
        print("\nNo Direct Train Available.")

In [11]:
while True:

    print("\n===== TRAIN ENQUIRY SYSTEM =====")

    source = input("Enter Source Station : ")
    destination = input(
        "Enter Destination Station : "
    )

    train_enquiry(source, destination)

    choice = input(
        "\nDo you want to search again? (Y/N): "
    )

    if choice.upper() != 'Y':
        print("Thank You!")
        break


===== TRAIN ENQUIRY SYSTEM =====


Enter Source Station :  mumbai
Enter Destination Station :  pune



No Direct Train Available.



Do you want to search again? (Y/N):  n


Thank You!
